# BPSD — BCPS pipeline (behavior-consistent popularity proxies)

Stages 1-2 (LightGCN backbone, behavioural profiles) are unchanged from the
original pipeline. Stage 3 (PPD's `p_i, r_ui, b_ui`) is replaced with BCPS:
four popularity signals computed only from raw interactions + metadata +
timestamps -- the same source `q_u`/`q_i` come from -- each tied to a named
behavioural mechanism instead of one generic embedding-space residual. Stage
4 is rebuilt on top of these: a fixed global basis (one direction per
mechanism, built by sequential Gram-Schmidt so `d_1` stays exactly the
mainstream-affinity direction) plus behaviour-conditioned gates that decide
how much of each fixed direction to remove per user/item.

Cells:
 1. Config + splits
 2. LightGCN backbone (Stage 1)
 3. Behavioural proxies from metadata (Stage 2)
 4. BCPS popularity proxies (Stage 3')
 5. BCPS popularity subspace + training (Stage 4')
 6. Diagnostics: identifiability, gating ablation, gate-spread check


In [ ]:
# ============================================================================
# CELL 1 — CONFIG AND SPLITS
# ============================================================================
import os, sys, time, math, json, random, collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

RAW_DATA_DIR = '/kaggle/input/datasets/tejasdeshmukh001/movielens'   # ratings.dat, movies.dat
WORKING_DIR  = '/kaggle/working/dataset/ml-1M'
CKPT_DIR     = '/kaggle/working/checkpoints'
os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

SEED = 2020
POSITIVE_RATING_THRESHOLD = 4.0   # PPAC uses `rating > 3`
TEST_HOLDOUT_PER_USER     = 30    # PPAC's TOP_K
BALANCE_PER_ITEM          = None  # None -> derive it from your own holdout pool (see below)
VAL_HOLDOUT_PER_USER      = 20    # only used when STRICT_PPAC = False

# STRICT_PPAC = True  -> reproduce the paper's released protocol exactly:
#                        no validation split, checkpoint selected by NDCG@50 on the
#                        balanced test set (this is literally what run_MF.py does).
#                        Use this ONLY for the head-to-head number against Table 2.
# STRICT_PPAC = False -> carve a validation split and select on it. Honest, but the
#                        absolute numbers land a little below the paper's.
STRICT_PPAC = False

EVAL_ON_BALANCED = True   # True  -> balanced (intervened) test set  == paper's numbers
                          # False -> the raw 30-per-user holdout     == biased test set

TOP_KS = [20, 50, 100]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)


def build_splits():
    """Reproduce PPAC's ml-1M split, plus its balanced (intervened) test set.

    PPAC (run_MF.py::create_train_and_test):
      * positives are ratings > 3
      * every user with MORE than 30 positives contributes exactly 30 to the test pool
      * every other user contributes all of their positives to train
      * NO global minimum-interaction user filter
      * NO validation split

    Balanced (intervened) test set: from the test pool, fix a per-item quota n,
    keep every item with at least n held-out interactions, and subsample exactly n
    of them. Every retained item then contributes the same number of test
    interactions, which is the property the protocol needs.

    n is not a free constant. Retained interactions = n * |{i : c_i >= n}|, which
    has an interior maximum: small n throws away interactions from popular items,
    large n throws away items entirely. BALANCE_PER_ITEM = None picks the argmax
    from THIS split's own item counts, so nothing is inherited from anywhere.

    Built entirely from ratings.dat and movies.dat. No external split files.
    """
    rng = random.Random(SEED)
    by_user = collections.defaultdict(list)
    rating_of = {}
    ts_of = {}

    with open(os.path.join(RAW_DATA_DIR, 'ratings.dat'), encoding='latin-1') as fh:
        for line in fh:
            u, i, r, t = line.strip().split('::')
            u, r, t = int(u), float(r), int(t)
            if r >= POSITIVE_RATING_THRESHOLD:
                by_user[u].append(i)
                rating_of[(u, i)] = r
                ts_of[(u, i)] = t

    train, val, test_pool = (collections.defaultdict(list) for _ in range(3))
    for u, items in by_user.items():
        items = list(items)
        rng.shuffle(items)
        need = TEST_HOLDOUT_PER_USER + (0 if STRICT_PPAC else VAL_HOLDOUT_PER_USER)
        if len(items) > need:
            test_pool[u] = items[:TEST_HOLDOUT_PER_USER]
            if not STRICT_PPAC:
                val[u] = items[TEST_HOLDOUT_PER_USER:need]
            train[u] = items[need:]
        else:
            train[u] = items

    # --- balanced / intervened sets -----------------------------------------
    def _balance(pool, quota=None):
        """Keep every item with >= n held-out interactions, subsample exactly n.

        n defaults to the argmax of retained(n) = n * |{i : c_i >= n}|, derived
        from this pool's own item counts. Applied to the validation pool as well
        as the test pool, so selection and reporting see the same kind of set.
        """
        hits = collections.defaultdict(list)
        for u, items in pool.items():
            for i in items:
                hits[i].append(u)
        counts = np.array(sorted(len(v) for v in hits.values()))
        retained = counts[::-1] * (np.arange(len(counts)) + 1)
        q = int(quota) if quota else int(counts[::-1][retained.argmax()])
        out, n_items = collections.defaultdict(list), 0
        for i, users in hits.items():
            if len(users) >= q:
                n_items += 1
                for u in rng.sample(users, q):
                    out[u].append(i)
        return out, q, n_items

    balanced, quota, n_bal_items = _balance(test_pool, BALANCE_PER_ITEM)

    print(f'[split] users={len(by_user)}  train_inter={sum(map(len, train.values()))}')
    print(f'[split] raw test: {len(test_pool)} users, {sum(map(len, test_pool.values()))} interactions')
    print(f'[split] balanced test: {len(balanced)} users, {sum(map(len, balanced.values()))} '
          f'interactions over {n_bal_items} items @ {quota} each')
    if not STRICT_PPAC:
        print(f'[split] val: {len(val)} users, {sum(map(len, val.values()))} interactions')

    def dump(name, recs):
        with open(os.path.join(WORKING_DIR, name), 'w', encoding='utf-8') as fh:
            for u, items in recs.items():
                for i in items:
                    fh.write(f'{u}::{i}::{rating_of[(u, i)]:.1f}::{ts_of[(u, i)]}\n')

    dump('ratings.train', train)
    dump('ratings.test', test_pool)
    dump('balance_ratings.test', balanced)
    if not STRICT_PPAC:
        dump('ratings.val', val)
        bal_val, vq, vn = _balance(val)
        print(f'[split] balanced val: {len(bal_val)} users, '
              f'{sum(map(len, bal_val.values()))} interactions over {vn} items @ {vq} each')
        dump('balance_ratings.val', bal_val)


def read_splits():
    """Load the splits and remap to compact 0-based ids."""
    item_map = {}
    with open(os.path.join(RAW_DATA_DIR, 'movies.dat'), encoding='latin-1') as fh:
        for idx, line in enumerate(fh):
            item_map[line.split('::')[0]] = idx

    def load(name):
        recs = collections.defaultdict(list)
        path = os.path.join(WORKING_DIR, name)
        if not os.path.exists(path):
            return recs
        with open(path, encoding='utf-8') as fh:
            for line in fh:
                u, i, _, _ = line.strip().split('::')
                recs[int(u)].append(item_map[i])
        return recs

    raw_train = load('ratings.train')
    user_map = {raw: new for new, raw in enumerate(sorted(raw_train))}
    train = collections.defaultdict(list, {user_map[u]: v for u, v in raw_train.items()})
    train_items = {i for v in train.values() for i in v}

    def rankable(name):
        out = collections.defaultdict(list)
        for u, items in load(name).items():
            if u not in user_map:
                continue
            keep = [i for i in items if i in train_items]
            if keep:
                out[user_map[u]] = keep
        return out

    test = rankable('balance_ratings.test' if EVAL_ON_BALANCED else 'ratings.test')
    val = rankable('balance_ratings.val' if EVAL_ON_BALANCED else 'ratings.val')
    if STRICT_PPAC:
        val = test          # paper protocol: selection happens on the eval set
    params = {'num_users': len(user_map), 'num_items': len(item_map)}
    return train, val, test, user_map, item_map, params


set_seed(SEED)
build_splits()
train_records, val_records, test_records, user_map, item_map, params = read_splits()
NUM_USERS, NUM_ITEMS = params['num_users'], params['num_items']
print(f'[data] users={NUM_USERS} items={NUM_ITEMS} '
      f'train={sum(map(len, train_records.values()))} '
      f'val_users={len(val_records)} test_users={len(test_records)}')


In [ ]:
# ============================================================================
# CELL 2 — STAGE 1: LIGHTGCN BACKBONE
# ============================================================================
LATENT_DIM = 64
N_LAYERS   = 3
LR         = 1e-3       # paper reports 0.01; 1e-3 is the released-code default
REG_WEIGHT = 1e-4
BATCH_SIZE = 8192
EPOCHS     = 2000
PATIENCE   = 50


def build_sparse_adj(train_recs, num_users, num_items):
    n = num_users + num_items
    us, it = [], []
    for u, items in train_recs.items():
        us.extend([u] * len(items))
        it.extend([i + num_users for i in items])
    us = torch.tensor(us, dtype=torch.long)
    it = torch.tensor(it, dtype=torch.long)
    src = torch.cat([us, it]); dst = torch.cat([it, us])
    deg = torch.zeros(n).scatter_add_(0, dst, torch.ones(dst.numel()))
    dis = deg.clamp(min=1.0).pow(-0.5)
    return torch.sparse_coo_tensor(
        torch.stack([dst, src]), dis[dst] * dis[src], (n, n)).coalesce()


def propagate(adj, e_u0, e_i0, n_layers=N_LAYERS):
    """LightGCN readout: mean over layers 0..L (alpha_l = 1/(L+1))."""
    x = torch.cat([e_u0, e_i0], dim=0)
    layers = [x]
    for _ in range(n_layers):
        x = torch.sparse.mm(adj, x)
        layers.append(x)
    out = torch.stack(layers, dim=1).mean(dim=1)
    return torch.split(out, [e_u0.shape[0], e_i0.shape[0]], dim=0)


def sample_triplets(train_recs, num_items):
    users, pos = [], []
    for u, items in train_recs.items():
        users.extend([u] * len(items)); pos.extend(items)
    users = np.asarray(users, dtype=np.int64); pos = np.asarray(pos, dtype=np.int64)
    psets = {u: set(v) for u, v in train_recs.items()}
    neg = np.random.randint(0, num_items, size=len(users), dtype=np.int64)
    bad = np.fromiter((n in psets[u] for u, n in zip(users, neg)), bool, len(users))
    while bad.any():
        idx = np.flatnonzero(bad)
        neg[idx] = np.random.randint(0, num_items, size=len(idx), dtype=np.int64)
        bad[idx] = np.fromiter((neg[j] in psets[users[j]] for j in idx), bool, len(idx))
    return torch.from_numpy(users), torch.from_numpy(pos), torch.from_numpy(neg)


def recall_ndcg(ground_truth, ranked, k):
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    rec, ndcg = [], []
    for truth, row in zip(ground_truth, ranked):
        ts = set(truth)
        hits = np.fromiter((i in ts for i in row[:k]), np.float32, k)
        rec.append(hits.sum() / len(ts))
        idcg = disc[:min(len(ts), k)].sum()
        ndcg.append(float((hits * disc).sum()) / idcg if idcg > 0 else 0.0)
    return float(np.mean(rec)), float(np.mean(ndcg))


@torch.no_grad()
def evaluate(e_u, e_i, eval_recs, exclude_recs, item_pop, ks=TOP_KS):
    users = sorted(u for u, v in eval_recs.items() if v)
    max_k = max(ks)
    ranked_all = []
    for s in range(0, len(users), 512):
        batch = users[s:s + 512]
        scores = e_u[batch] @ e_i.T
        for r, u in enumerate(batch):
            ex = exclude_recs.get(u, [])
            if ex:
                scores[r, torch.tensor(ex, dtype=torch.long, device=scores.device)] = -torch.inf
        ranked_all.append(torch.topk(scores, k=max_k, dim=1).indices.cpu().numpy())
    ranked = np.concatenate(ranked_all, 0)
    truth = [eval_recs[u] for u in users]

    out = {}
    for k in ks:
        r, n = recall_ndcg(truth, ranked, k)
        out[k] = {'recall': r, 'ndcg': n, 'arp': float(item_pop[ranked[:, :k]].mean())}
    tail = set(np.flatnonzero(item_pop <= np.median(item_pop)).tolist())
    tt = [[i for i in t if i in tail] for t in truth]
    keep = [j for j, t in enumerate(tt) if t]
    out['tail_recall@50'] = recall_ndcg([tt[j] for j in keep], ranked[keep], 50)[0] if keep else 0.0
    return out


class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, dim=LATENT_DIM):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, dim)
        self.item_embedding = nn.Embedding(num_items, dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def readout(self, adj):
        return propagate(adj, self.user_embedding.weight, self.item_embedding.weight)


def item_popularity_from(train_recs, num_items):
    c = np.zeros(num_items, dtype=np.float32)
    for items in train_recs.values():
        c[np.asarray(items, dtype=np.int64)] += 1.0
    return c / max(float(c.max()), 1.0)


def train_backbone():
    set_seed(SEED)
    adj = build_sparse_adj(train_records, NUM_USERS, NUM_ITEMS).to(DEVICE)
    model = LightGCN(NUM_USERS, NUM_ITEMS).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    item_pop = item_popularity_from(train_records, NUM_ITEMS)
    ckpt = os.path.join(CKPT_DIR, 'lightgcn-ml1m.pt')

    best, best_ep = -np.inf, -1
    for epoch in range(EPOCHS):
        model.train(); t0 = time.time()
        u, p, n = sample_triplets(train_records, NUM_ITEMS)
        perm = torch.randperm(len(u)); u, p, n = u[perm], p[perm], n[perm]
        tot = 0.0; nb = 0
        for s in range(0, len(u), BATCH_SIZE):
            ub = u[s:s+BATCH_SIZE].to(DEVICE)
            pb = p[s:s+BATCH_SIZE].to(DEVICE)
            nb_ = n[s:s+BATCH_SIZE].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            eu, ei = model.readout(adj)
            pos = (eu[ub] * ei[pb]).sum(1)
            neg = (eu[ub] * ei[nb_]).sum(1)
            rank = F.softplus(neg - pos).mean()
            ego = (model.user_embedding(ub).pow(2).sum()
                   + model.item_embedding(pb).pow(2).sum()
                   + model.item_embedding(nb_).pow(2).sum()) / (2.0 * len(ub))
            loss = rank + REG_WEIGHT * ego
            loss.backward(); opt.step()
            tot += loss.item(); nb += 1

        model.eval()
        with torch.no_grad():
            eu, ei = model.readout(adj)
            m = evaluate(eu, ei, val_records, train_records, item_pop)
        if m[50]['ndcg'] > best:
            best, best_ep = m[50]['ndcg'], epoch
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()}, ckpt)
        print(f'Epoch [{epoch+1}/{EPOCHS}] Loss {tot/nb:.4f} '
              f'Recall@50 {m[50]["recall"]:.4f} NDCG@50 {m[50]["ndcg"]:.4f} '
              f'ARP@50 {m[50]["arp"]:.4f} {time.time()-t0:.1f}s')
        if epoch - best_ep >= PATIENCE:
            print(f'Early stop at {epoch+1}; best epoch {best_ep+1}'); break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)['model_state_dict'])
    torch.save({'adjacency': adj.cpu().coalesce(),
                'num_users': NUM_USERS, 'num_items': NUM_ITEMS},
               '/kaggle/working/sparse_adj_matrix.pt')
    model.eval()
    with torch.no_grad():
        eu, ei = model.readout(adj)
        excl = train_records if STRICT_PPAC else {
            u: train_records.get(u, []) + val_records.get(u, []) for u in train_records}
        m = evaluate(eu, ei, test_records, excl, item_pop)
    print('\n--- BASELINE LightGCN (test) ---')
    for k in TOP_KS:
        print(f'Recall@{k} {m[k]["recall"]:.4f}  NDCG@{k} {m[k]["ndcg"]:.4f}  ARP@{k} {m[k]["arp"]:.4f}')
    return model, adj, item_pop


backbone, sparse_adj, item_pop = train_backbone()


In [ ]:
# ============================================================================
# CELL 3 — STAGE 2: BEHAVIOURAL PROFILES (§7)
# ============================================================================
# Deviation from the old notebook: categories and content vectors come from
# movies.dat genres, NOT from KMeans on the LightGCN item embeddings. Clustering
# the backbone's own embeddings makes q_u a function of the representation being
# debiased -- the profiles then re-encode the popularity signal they are supposed
# to be independent of, and §7 says these come from metadata.
from scipy.stats import spearmanr, entropy
from sklearn.cluster import KMeans

N_CATEGORIES  = 20
CATEGORY_MODE = 'kmeans_genre'   # 'kmeans_genre' | 'primary_genre'
WINDOW_DAYS   = 30               # §7.3 uses *time* windows, not fixed-count chunks
MIN_WINDOW_INTERACTIONS = 3
LOYALTY_WEIGHT = 'count'         # 'count' matches §7.6 with w_ui = #interactions
DECONFOUND = False               # §7.8 -- OFF, as requested
PROXY_COLS = ['diversity', 'temporal_stability', 'exploration', 'cross_category', 'loyalty']


def build_item_metadata(item_map, n_categories=N_CATEGORIES):
    """Genre multi-hot content vectors + one category per item, from movies.dat."""
    genres_of = {}
    vocab = {}
    with open(os.path.join(RAW_DATA_DIR, 'movies.dat'), encoding='latin-1') as fh:
        for line in fh:
            mid, _, gstr = line.strip().split('::')
            gs = gstr.split('|')
            genres_of[item_map[mid]] = gs
            for g in gs:
                vocab.setdefault(g, len(vocab))

    content = np.zeros((len(item_map), len(vocab)), dtype=np.float32)
    for i, gs in genres_of.items():
        for g in gs:
            content[i, vocab[g]] = 1.0
    content /= np.maximum(np.linalg.norm(content, axis=1, keepdims=True), 1e-9)

    if CATEGORY_MODE == 'primary_genre':
        freq = collections.Counter(g for gs in genres_of.values() for g in gs)
        cat = np.array([vocab[min(genres_of[i], key=lambda g: freq[g])]
                        for i in range(len(item_map))], dtype=np.int64)
    else:
        km = KMeans(n_clusters=min(n_categories, len(item_map)), random_state=42, n_init=10)
        cat = km.fit_predict(content).astype(np.int64)
    return content, cat, int(cat.max()) + 1


def user_profiles(interactions, content, category, n_cats):
    """§7.2-7.7. interactions: DataFrame[user, item, rating, timestamp]."""
    recs = []
    for uid, grp in interactions.groupby('user', sort=True):
        items = grp['item'].to_numpy()
        if len(items) < 2:
            continue
        cats = category[items]

        # §7.2 diversity: mean pairwise (1 - cos) over CONTENT vectors
        V = content[items]
        S = V @ V.T
        iu = np.triu_indices(len(items), k=1)
        diversity = float((1.0 - S[iu]).mean())

        # §7.5 cross-category reach: Shannon entropy of P(c|u)
        counts = np.bincount(cats, minlength=n_cats).astype(np.float64)
        probs = counts[counts > 0] / counts.sum()
        cross = float(entropy(probs))

        # §7.4 exploration: fraction of items outside the user's top-3 categories
        top3 = set(np.argsort(-counts)[:3].tolist())
        exploration = float(np.mean([c not in top3 for c in cats]))

        # §7.6 loyalty
        w = (grp['rating'].to_numpy(np.float64) if LOYALTY_WEIGHT == 'rating'
             else np.ones(len(items)))
        agg = np.bincount(pd.factorize(items)[0], weights=w)
        loyalty = float(((agg / agg.sum()) ** 2).sum())

        # §7.3 temporal stability: Spearman between the FULL n_cats-length category
        # histograms of consecutive TIME windows.
        ts = grp['timestamp'].to_numpy(np.int64)
        order = np.argsort(ts)
        ts_o, cats_o = ts[order], cats[order]
        span = WINDOW_DAYS * 86400
        wins, cur, start = [], [], ts_o[0]
        for t, c in zip(ts_o, cats_o):
            if t - start > span and len(cur) >= MIN_WINDOW_INTERACTIONS:
                wins.append(cur); cur, start = [], t
            cur.append(c)
        if len(cur) >= MIN_WINDOW_INTERACTIONS:
            wins.append(cur)
        if len(wins) < 2:                       # fallback: equal-count halves
            half = len(cats_o) // 2
            wins = [cats_o[:half].tolist(), cats_o[half:].tolist()] if half >= 1 else []
        rhos = []
        for a, b in zip(wins[:-1], wins[1:]):
            ha = np.bincount(np.asarray(a), minlength=n_cats).astype(np.float64)
            hb = np.bincount(np.asarray(b), minlength=n_cats).astype(np.float64)
            if ha.std() > 0 and hb.std() > 0:
                rho = spearmanr(ha, hb).statistic
                if not np.isnan(rho):
                    rhos.append(rho)
        temporal = float(np.mean(rhos)) if rhos else 0.0

        recs.append({'user': uid, 'diversity': diversity, 'temporal_stability': temporal,
                     'exploration': exploration, 'cross_category': cross,
                     'loyalty': loyalty, '_degree': len(items)})

    df = pd.DataFrame(recs)
    for c in PROXY_COLS:                                       # §7.7 min-max
        lo, hi = df[c].min(), df[c].max()
        df[c] = (df[c] - lo) / (hi - lo + 1e-8)
    if DECONFOUND:                                             # §7.8 -- OFF
        bins = pd.qcut(df['_degree'], q=10, labels=False, duplicates='drop')
        for c in PROXY_COLS:
            df[c] = df[c] - df.groupby(bins)[c].transform('mean')
    return df.drop(columns=['_degree'])


def build_profiles():
    inter = []
    inv_item = {v: k for k, v in item_map.items()}
    ts_lookup = {}
    with open(os.path.join(RAW_DATA_DIR, 'ratings.dat'), encoding='latin-1') as fh:
        for line in fh:
            u, i, r, t = line.strip().split('::')
            ts_lookup[(int(u), i)] = (float(r), int(t))
    inv_user = {v: k for k, v in user_map.items()}
    for u, items in train_records.items():
        ru = inv_user[u]
        for i in items:
            r, t = ts_lookup[(ru, inv_item[i])]
            inter.append((u, i, r, t))
    df = pd.DataFrame(inter, columns=['user', 'item', 'rating', 'timestamp'])

    content, category, n_cats = build_item_metadata(item_map)
    q_u_df = user_profiles(df, content, category, n_cats)
    print(f'[stage2] q_u for {len(q_u_df)}/{NUM_USERS} users, {n_cats} categories')

    # §7.9 q_i = mean of q_u over N(i); unseen items get the COLUMN MEAN, not 0
    # (0 is the minimum after min-max normalisation, not a neutral value).
    q_i_df = (df[['user', 'item']].merge(q_u_df, on='user', how='inner')
              .groupby('item')[PROXY_COLS].mean().reset_index())
    q_i_df = pd.DataFrame({'item': np.arange(NUM_ITEMS)}).merge(q_i_df, on='item', how='left')
    q_i_df[PROXY_COLS] = q_i_df[PROXY_COLS].fillna(q_i_df[PROXY_COLS].mean())

    q_u = np.tile(q_u_df[PROXY_COLS].mean().to_numpy(np.float32), (NUM_USERS, 1))
    q_u[q_u_df['user'].to_numpy(np.int64)] = q_u_df[PROXY_COLS].to_numpy(np.float32)
    q_i = q_i_df[PROXY_COLS].to_numpy(np.float32)

    q_u_df.to_csv('/kaggle/working/q_u_profiles.csv', index=False)
    q_i_df.to_csv('/kaggle/working/q_i_profiles.csv', index=False)
    return torch.tensor(q_u, device=DEVICE), torch.tensor(q_i, device=DEVICE), df


q_u, q_i, interactions_df = build_profiles()


In [ ]:
# ============================================================================
# CELL 4 — STAGE 3': BEHAVIOR-CONSISTENT POPULARITY SIGNALS (BCPS)
#
# Replaces PPD's (p_i, r_ui, b_ui). Those were computed from e_u_final0/
# e_i_final0 -- the same embedding space Stage 4 debiases -- so the learned
# direction partly re-encoded the backbone's own geometry, and b_ui is a raw,
# high-variance per-edge residual while q_u/q_i are smooth, activity-
# deconfounded vectors. Combining the two fought the behavioural gating
# instead of cooperating with it.
#
# BCPS computes popularity signals EXCLUSIVELY from raw interactions +
# metadata + timestamps -- the same source q_u/q_i come from -- and
# decomposes "popularity" into four NAMED behavioural mechanisms instead of
# one generic residual, so each subspace direction built in Stage 4' has a
# testable meaning instead of being an arbitrary residual axis.
#
#   rho_i        static global popularity (log-compressed, normalised)
#   rho_i(t)     item's popularity SHARE within a global calendar window
#                (needs timestamps; global windows, not per-user, since an
#                item's popularity trajectory needs one shared clock)
#   rho_i^cat    item's popularity share WITHIN its own category
#                ("big fish, small pond"; reuses Stage 2's genre categories)
#   sigma_u      user's popularity-appetite baseline: mean rho_i of the items
#                u consumes, activity-deconfounded with the same §7.8 recipe
#                used for q_u
#
# Four interaction-level mechanism scores, one per popularity subspace mode:
#   beta_ui^(1)  mainstream affinity      = rho_i
#   beta_ui^(2)  temporal conformity      = rho_i(t_ui)
#   beta_ui^(3)  category dominance       = rho_i^cat
#   beta_ui^(4)  susceptibility-adjusted  = rho_i(t_ui) - sigma_u
#
# None of these touch e_u^(0)/e_i^(0) -- computed once from the raw train
# interactions, before Stage 4' ever looks at the backbone.
# ============================================================================
EPS = 1e-8

MECHANISM_NAMES = ['mainstream_affinity', 'temporal_conformity',
                    'category_dominance', 'susceptibility_adjusted']
N_MECH = len(MECHANISM_NAMES)
BCPS_WINDOW_DAYS = WINDOW_DAYS          # reuse Stage 2's window length (§7.3)
SUSCEPT_DECONFOUND = True               # deconfound sigma_u by activity group,
                                         # same recipe as §7.8 for q_u (kept
                                         # independent of Stage 2's own
                                         # DECONFOUND flag -- a raw popularity
                                         # MEAN is far more degree-confounded
                                         # than the §7 proxies are)


def build_bcps_proxies(df, item_map, category, n_users=NUM_USERS, n_items=NUM_ITEMS,
                        window_days=BCPS_WINDOW_DAYS,
                        deconfound_susceptibility=SUSCEPT_DECONFOUND):
    """All four mechanism scores, from raw interactions + metadata only.

    df: interactions_df (Stage 2's train-only DataFrame: user, item, rating,
        timestamp). category: (n_items,) int array from build_item_metadata.
    Returns (bcps_df, beta[E, N_MECH] float32 in [0,1], rho_i[n_items], sigma_u[n_users]).
    """
    user_ids = df['user'].to_numpy(np.int64)
    item_ids = df['item'].to_numpy(np.int64)

    # --- 2.1 static global popularity: rho_i ---------------------------------
    deg = np.bincount(item_ids, minlength=n_items).astype(np.float64)
    rho_raw = np.log1p(deg)
    rho_i = (rho_raw - rho_raw.min()) / (rho_raw.max() - rho_raw.min() + EPS)

    # --- 2.2 time-windowed popularity share: rho_i(t) -------------------------
    # GLOBAL calendar windows (one shared clock for every item), unlike Stage
    # 2's per-user windows which start at each user's own first interaction.
    ts = df['timestamp'].to_numpy(np.int64)
    span = window_days * 86400
    win_id = ((ts - ts.min()) // span).astype(np.int64)
    n_wins = int(win_id.max()) + 1
    flat = win_id * n_items + item_ids
    flat_counts = np.bincount(flat, minlength=n_wins * n_items).astype(np.float64)
    win_totals = np.maximum(flat_counts.reshape(n_wins, n_items).sum(axis=1), EPS)
    rho_it = flat_counts.reshape(n_wins, n_items) / win_totals[:, None]
    rho_t_edge_raw = rho_it[win_id, item_ids]
    rho_t_edge = (rho_t_edge_raw - rho_t_edge_raw.min()) / (
        rho_t_edge_raw.max() - rho_t_edge_raw.min() + EPS)

    # --- 2.3 category-local popularity: rho_i^cat ("big fish, small pond") ----
    n_cats = int(category.max()) + 1
    cat_totals = np.bincount(category, weights=deg, minlength=n_cats)
    rho_cat_raw = deg / np.maximum(cat_totals[category], EPS)
    rho_cat = (rho_cat_raw - rho_cat_raw.min()) / (rho_cat_raw.max() - rho_cat_raw.min() + EPS)

    # --- 3. user popularity-appetite baseline: sigma_u ------------------------
    sigma_sum = np.bincount(user_ids, weights=rho_i[item_ids], minlength=n_users)
    sigma_cnt = np.maximum(np.bincount(user_ids, minlength=n_users), 1)
    sigma_u = sigma_sum / sigma_cnt

    if deconfound_susceptibility:
        deg_u = pd.Series(np.bincount(user_ids, minlength=n_users))
        bins = pd.qcut(deg_u, q=10, labels=False, duplicates='drop')
        grp_mean = pd.Series(sigma_u).groupby(bins).transform('mean')
        sigma_u = sigma_u - grp_mean.to_numpy()

    # --- 4. four interaction-level mechanism scores, in [0,1] -----------------
    beta_1 = rho_i[item_ids]                                    # mainstream affinity
    beta_2 = rho_t_edge                                          # temporal conformity
    beta_3 = rho_cat[item_ids]                                  # category dominance
    beta_4_raw = rho_t_edge - sigma_u[user_ids]                 # susceptibility-adjusted
    beta_4 = (beta_4_raw - beta_4_raw.min()) / (beta_4_raw.max() - beta_4_raw.min() + EPS)

    beta = np.clip(np.stack([beta_1, beta_2, beta_3, beta_4], axis=1), 0.0, 1.0).astype(np.float32)

    out = df[['user', 'item']].copy()
    for k, name in enumerate(MECHANISM_NAMES):
        out[name] = beta[:, k]
    return out, beta, rho_i, sigma_u


# category comes from Stage 2's KMeans-on-genre assignment; recomputed here
# (deterministic, same random_state=42) so this cell stays self-contained.
_content_bcps, category_bcps, _n_cats_bcps = build_item_metadata(item_map)

bcps_df, beta_np, rho_i_arr, sigma_u_arr = build_bcps_proxies(
    interactions_df, item_map, category_bcps)
bcps_df.to_csv('/kaggle/working/stage3_bcps_interactions.csv', index=False)

edge_u = torch.tensor(bcps_df['user'].to_numpy(np.int64), device=DEVICE)
edge_i = torch.tensor(bcps_df['item'].to_numpy(np.int64), device=DEVICE)
beta_ui = torch.tensor(beta_np, device=DEVICE)     # (E, N_MECH)

print('[stage3\'] BCPS mechanism scores (min / mean / max):')
for k, name in enumerate(MECHANISM_NAMES):
    col = beta_np[:, k]
    print(f'  {name:<24} [{col.min():.4f}, {col.mean():.4f}, {col.max():.4f}]')

corr = np.corrcoef(beta_np.T)
print('\n[stage3\'] mechanism correlation matrix (should NOT all be ~1.0 -- if they')
print('           are, the mechanisms are redundant and Stage 4\' will inherit the')
print('           same collapse a single generic weight would produce):')
header = '           ' + ''.join(f'{n[:10]:>12}' for n in MECHANISM_NAMES)
print(header)
for name, r in zip(MECHANISM_NAMES, corr):
    print(f'{name[:10]:>10} ' + ''.join(f'{v:>12.3f}' for v in r))


In [ ]:
# ============================================================================
# CELL 5 — STAGE 4': BCPS POPULARITY SUBSPACE
#
# Architecture: a FIXED global basis (K directions, built once from the
# BCPS mechanism scores, not free-trained) plus a small behaviour-conditioned
# GATING network that decides how much of each fixed direction to remove per
# user/item. D is never a free nn.Parameter: with the backbone frozen, BPR
# gradients on a free D only rotate it into directions the embeddings barely
# use, so the subtraction removes nothing. Only the gates are trained.
#
# Basis construction: each named mechanism gets its own centroid-difference
# direction
#     d_k_raw = c_pop,k - phi * c_pref,k
# where c_pop,k / c_pref,k are beta_ui^(k)-weighted / (1-beta_ui^(k))-weighted
# means of the neighbouring item embeddings. Built in PRIORITY order
# (mechanism 1 first) and Gram-Schmidt deflated against every higher-priority
# direction already built, so d_1 is EXACTLY the mainstream-affinity
# direction and d_2..d_4 are each mechanism's own signal with only the
# overlap already captured by earlier modes removed -- not an arbitrary
# rotation. The weight itself differs across k (four different mechanism
# scores), not just a partition of users being averaged, which is what lets
# the K centroids stay distinct instead of collapsing to the same mean.
# ============================================================================
NUM_MODES     = N_MECH   # K = 4 named mechanisms
PHI           = 1.0
ALPHA         = 0.3      # removal strength; sweep this
GATE_TEMP     = 1.0
LAMBDA_ORTH   = 1e-3     # directions are orthonormal by construction; kept small
LAMBDA_REG    = 1e-4
DEBIAS_LR     = 1e-2
DEBIAS_EPOCHS = 300
EVAL_EVERY    = 10
DEBIAS_PATIENCE = 10


def build_popularity_basis(e_i0, edge_i, beta, num_modes, phi=PHI):
    """Named, sequentially-deflated popularity directions from BCPS mechanism
    scores. beta: (E, N_MECH) in [0,1]. Returns (num_modes, d), orthonormal rows."""
    with torch.no_grad():
        n_mech = beta.shape[1]
        use_mech = min(num_modes, n_mech)

        raw_dirs = []
        for k in range(use_mech):
            b_k = beta[:, k]
            w_pop = torch.zeros(e_i0.shape[0], device=e_i0.device).index_add(0, edge_i, b_k)
            w_pre = torch.zeros(e_i0.shape[0], device=e_i0.device).index_add(0, edge_i, 1.0 - b_k)
            c_pop = (w_pop.unsqueeze(1) * e_i0).sum(0) / (w_pop.sum() + EPS)
            c_pre = (w_pre.unsqueeze(1) * e_i0).sum(0) / (w_pre.sum() + EPS)
            raw_dirs.append(F.normalize(c_pop - phi * c_pre, dim=0))
        raw_stack = torch.stack(raw_dirs, dim=0)
        pre_cos = raw_stack @ raw_stack.t()
        print(f'   [basis] pre-deflation cosine between mechanism directions:')
        for i, name in enumerate(MECHANISM_NAMES[:use_mech]):
            print('      ' + f'{name[:22]:<22}' +
                  ''.join(f'{pre_cos[i, j].item():>8.3f}' for j in range(use_mech)))

        dirs = []
        for d in raw_dirs:
            for prev in dirs:
                d = d - (d @ prev) * prev
            if d.norm() < 1e-6:
                print('   [basis] WARNING: a mechanism direction collapsed to ~0 after '
                      'deflation (fully explained by higher-priority mechanisms); '
                      'falling back to a random orthogonal complement.')
                d = torch.randn_like(d)
                for prev in dirs:
                    d = d - (d @ prev) * prev
            dirs.append(F.normalize(d, dim=0))
        basis = torch.stack(dirs, dim=0)

        if num_modes > use_mech:                          # residual PCA axes
            b0 = beta[:, 0]
            w = torch.zeros(e_i0.shape[0], device=e_i0.device).index_add(0, edge_i, b0)
            w = (w / (w.sum() + EPS)).unsqueeze(1)
            Xc = (e_i0 - (w * e_i0).sum(0, keepdim=True)) * w.sqrt()
            for d in dirs:
                Xc = Xc - (Xc @ d).unsqueeze(1) * d
            _, S, V = torch.linalg.svd(Xc, full_matrices=False)
            basis = torch.cat([basis, V[:num_modes - use_mech]], dim=0)
            print(f'   [basis] +{num_modes - use_mech} residual PCA axes, '
                  f'spectrum {[round(x, 3) for x in S[:num_modes - use_mech + 1].tolist()]}')

        basis = torch.linalg.qr(basis.t(), mode='reduced').Q.t()   # defensive cleanup only
        return basis


class BPSD(nn.Module):
    """Shared popularity subspace with behaviour-conditioned soft removal."""

    def __init__(self, basis, n_proxies=5, alpha=ALPHA,
                 temp=GATE_TEMP, uniform_gates=False):
        super().__init__()
        self.register_buffer('basis', basis)          # (K, d), fixed, orthonormal
        self.K = basis.shape[0]
        self.alpha, self.temp = alpha, temp
        self.uniform_gates = uniform_gates
        self.user_gate = nn.Linear(n_proxies, self.K)   # W_u, b_u
        self.item_gate = nn.Linear(n_proxies, self.K)   # W_i, b_i
        for lin in (self.user_gate, self.item_gate):
            nn.init.xavier_uniform_(lin.weight); nn.init.zeros_(lin.bias)

    def has_trainable_gates(self):
        return (not self.uniform_gates) and self.K > 1

    def gates(self, q_u, q_i):                          # §9.1
        if self.uniform_gates:
            return (q_u.new_full((q_u.shape[0], self.K), 1.0 / self.K),
                    q_i.new_full((q_i.shape[0], self.K), 1.0 / self.K))
        return (torch.softmax(self.user_gate(q_u) / self.temp, dim=-1),
                torch.softmax(self.item_gate(q_i) / self.temp, dim=-1))

    def forward(self, e_u0, e_i0, q_u, q_i):
        g_u, g_i = self.gates(q_u, q_i)
        D = self.basis
        rm_u = self.alpha * (((e_u0 @ D.t()) * g_u) @ D)   # §9.10 with removal strength
        rm_i = self.alpha * (((e_i0 @ D.t()) * g_i) @ D)
        I = torch.eye(self.K, device=D.device, dtype=D.dtype)
        orth = (D @ D.t() - I).pow(2).sum()
        return e_u0 - rm_u, e_i0 - rm_i, g_u, g_i, orth, (rm_u, rm_i)


def train_debiaser(num_modes=NUM_MODES, alpha=ALPHA, verbose=True, uniform_gates=False):
    set_seed(SEED)
    e_u_frozen = backbone.user_embedding.weight.detach().to(DEVICE)   # §13.2 frozen
    e_i_frozen = backbone.item_embedding.weight.detach().to(DEVICE)

    basis = build_popularity_basis(e_i_frozen, edge_i, beta_ui, num_modes)
    model = BPSD(basis, n_proxies=len(PROXY_COLS), alpha=alpha,
                 uniform_gates=uniform_gates).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=DEBIAS_LR)
    tag = f'bcps_K{num_modes}_a{alpha}' + ('_uniform' if uniform_gates else '')
    ckpt = f'/kaggle/working/bpsd_{tag}.pt'
    best, best_ep, stall = -np.inf, 0, 0

    n_epochs = DEBIAS_EPOCHS if model.has_trainable_gates() else EVAL_EVERY
    if not model.has_trainable_gates():
        print(f'   [{tag}] parameter-free variant; evaluating the fixed projection once.')

    for epoch in range(1, n_epochs + 1):
        model.train(); t0 = time.time()
        u, p, n = sample_triplets(train_records, NUM_ITEMS)
        perm = torch.randperm(len(u))
        u, p, n = u[perm].to(DEVICE), p[perm].to(DEVICE), n[perm].to(DEVICE)

        opt.zero_grad(set_to_none=True)
        eu0, ei0, g_u_, g_i_, orth, (rm_u, rm_i) = model(e_u_frozen, e_i_frozen, q_u, q_i)
        eu, ei = propagate(sparse_adj, eu0, ei0)

        bpr = torch.zeros((), device=DEVICE)
        for s in range(0, len(u), BATCH_SIZE):
            sl = slice(s, s + BATCH_SIZE)
            margin = (eu[u[sl]] * ei[p[sl]]).sum(1) - (eu[u[sl]] * ei[n[sl]]).sum(1)
            bpr = bpr + F.softplus(-margin).sum()
        bpr = bpr / len(u)

        reg = sum(prm.pow(2).sum() for prm in model.parameters())
        loss = bpr + LAMBDA_ORTH * orth + LAMBDA_REG * reg
        loss.backward(); opt.step()

        if epoch % EVAL_EVERY:
            continue
        model.eval()
        with torch.no_grad():
            eu0, ei0, *_ = model(e_u_frozen, e_i_frozen, q_u, q_i)
            eu, ei = propagate(sparse_adj, eu0, ei0)
            m = evaluate(eu, ei, val_records, train_records, item_pop)
        if m[50]['ndcg'] > best:
            best, best_ep, stall = m[50]['ndcg'], epoch, 0
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()}, ckpt)
        else:
            stall += 1
        if verbose:
            print(f'   [{tag}] ep {epoch:>3} loss {loss.item():.5f} '
                  f'removed|u| {rm_u.norm(dim=1).mean().item():.3f} '
                  f'gatestd {g_u_.std(0).mean().item():.3f} '
                  f'R@50 {m[50]["recall"]:.4f} N@50 {m[50]["ndcg"]:.4f} '
                  f'ARP {m[50]["arp"]:.4f} tail {m["tail_recall@50"]:.4f} '
                  f'{time.time()-t0:.1f}s')
        if stall >= DEBIAS_PATIENCE:
            break

    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE)['model_state_dict'])
    model.eval()
    with torch.no_grad():
        eu0, ei0, *_ = model(e_u_frozen, e_i_frozen, q_u, q_i)
        eu, ei = propagate(sparse_adj, eu0, ei0)
        m = evaluate(eu, ei, test_records, train_records, item_pop)
    return model, m


def row(name, m):
    print(f'{name:<28} R@50 {m[50]["recall"]:.4f}  N@50 {m[50]["ndcg"]:.4f}  '
          f'ARP@50 {m[50]["arp"]:.4f}  Tail@50 {m["tail_recall@50"]:.4f}')


# --- alpha sweep: find a usable operating point ------------------------------
print('=== alpha sweep, K=4 (BCPS mechanisms) ===')
bcps_runs = {a: train_debiaser(NUM_MODES, alpha=a, verbose=False)[1]
             for a in [0.1, 0.2, 0.3, 0.5, 1.0]}
for a, m in bcps_runs.items():
    row(f'BCPS (K=4, alpha={a})', m)

# --- gating ablation at the best alpha (set from the sweep above) -----------
# BEST_ALPHA = 0.3
# _, m_uniform = train_debiaser(NUM_MODES, alpha=BEST_ALPHA, uniform_gates=True)
# _, m_learned = train_debiaser(NUM_MODES, alpha=BEST_ALPHA)
# row('BCPS K=4 uniform gates', m_uniform); row('BCPS K=4 learned gates', m_learned)


In [ ]:
# ============================================================================
# CELL 6 — DIAGNOSTICS: IS THE BCPS SUBSPACE ACTUALLY IDENTIFIABLE?
#
# Closed-form checks, no training:
#
# 1. Does each learned direction d_k actually correlate with the mechanism it
#    was built from? R^2 between (item embedding projected onto d_k) and
#    (that item's own mean beta^(k)). Low R^2 for a mode means that mechanism
#    doesn't have a clean embedding-space correlate on this backbone -- treat
#    that mode as unidentified, not as "working", regardless of what it does
#    to Recall/NDCG. The basis is a fixed buffer, so this is a direct
#    correlation, not a probe requiring its own training loop.
#
# 2. Recall/ARP/tail-recall at matched removal strength (alpha) AND at
#    matched popularity level (ARP), for uniform vs. learned gates -- isolates
#    whether the behavioural gating is contributing anything on top of the
#    fixed BCPS basis itself.
# ============================================================================

# --- 1. identifiability: R^2 of (d_k^T e_i) vs item-level mean beta^(k) ------
with torch.no_grad():
    e_i_frozen = backbone.item_embedding.weight.detach().to(DEVICE)
    basis = build_popularity_basis(e_i_frozen, edge_i, beta_ui, NUM_MODES)
    proj = (e_i_frozen @ basis.t()).cpu().numpy()          # (n_items, N_MECH)

    edge_item = bcps_df['item'].to_numpy(np.int64)
    print(f'{"mechanism":<24}{"R^2(proj, mean beta)":>22}')
    for k, name in enumerate(MECHANISM_NAMES):
        item_beta_sum = np.bincount(edge_item, weights=beta_np[:, k], minlength=NUM_ITEMS)
        item_beta_cnt = np.maximum(np.bincount(edge_item, minlength=NUM_ITEMS), 1)
        item_beta_mean = item_beta_sum / item_beta_cnt
        present = item_pop > 0
        x, y = proj[present, k], item_beta_mean[present]
        if x.std() > 0 and y.std() > 0:
            r2 = np.corrcoef(x, y)[0, 1] ** 2
        else:
            r2 = 0.0
        print(f'{name:<24}{r2:>22.4f}')
    print('(low R^2 => that direction does not encode its intended mechanism on')
    print(' this backbone -- treat that mode as unidentified, not as "working")')


# --- 2. gating ablation: does behaviour-conditioning add anything on top of --
# --- the fixed BCPS basis itself? Compare uniform gates (g = 1/K for every ---
# --- node) against learned gates, at the same alpha. -------------------------
BEST_ALPHA = min(bcps_runs, key=lambda a: -bcps_runs[a][50]['ndcg'])
print(f'\n=== gating ablation at alpha={BEST_ALPHA} (best NDCG@50 from the sweep) ===')
_, m_uniform = train_debiaser(NUM_MODES, alpha=BEST_ALPHA, uniform_gates=True, verbose=False)
_, m_learned = train_debiaser(NUM_MODES, alpha=BEST_ALPHA, uniform_gates=False, verbose=False)
row('BCPS K=4 uniform gates', m_uniform)
row('BCPS K=4 learned gates', m_learned)
print('(if learned <= uniform here, the behavioural gating is not adding anything')
print(' on top of the fixed BCPS basis at this operating point -- the win, if any,')
print(' is coming from the proxy redesign itself, not from behaviour-conditioning)')


# --- 3. mode-collapse check on the trained model -----------------------------
# Confirms the four directions stay distinct (not just at construction time,
# which Cell 5 already checks) but also that the GATES spread mass across
# them differently for different users, rather than converging to the same
# distribution for everyone (the failure mode diagnosed against b_ui).
trained_model, _ = train_debiaser(NUM_MODES, alpha=BEST_ALPHA, verbose=False)
with torch.no_grad():
    g_u, g_i = trained_model.gates(q_u, q_i)
    print(f'\n[gate spread] per-mode std across users: '
          f'{[round(x, 4) for x in g_u.std(0).cpu().tolist()]}')
    print(f'[gate spread] per-mode std across items: '
          f'{[round(x, 4) for x in g_i.std(0).cpu().tolist()]}')
    print('(near-zero std for a mode means every user/item gets ~the same gate')
    print(' weight for that mode -- the behavioural profile is not distinguishing')
    print(' anyone on that axis)')
